<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:

from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List, Optional, Union

import pandas as pd
import yaml


DEFAULT_REGISTRY_PATH = "../../feature_store/registry/feature_registry.yaml"


class FeatureRegistryError(Exception):
    pass


def load_feature_registry(
    registry_path: Union[str, Path] = DEFAULT_REGISTRY_PATH
) -> Dict[str, Any]:
    registry_path = Path(registry_path).resolve()

    if not registry_path.exists():
        raise FileNotFoundError(f"Registry file not found: {registry_path}")

    with open(registry_path, "r", encoding="utf-8") as f:
        registry = yaml.safe_load(f)

    if not isinstance(registry, dict):
        raise FeatureRegistryError("Registry YAML did not load as a dictionary.")

    return registry


def _entity_map(registry: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    return {entity["name"]: entity for entity in registry.get("entities", [])}


def validate_registry(registry: Dict[str, Any]) -> List[str]:
    errors: List[str] = []

    required_top_level = ["registry_name", "registry_version", "entities", "feature_sets", "governance"]
    for key in required_top_level:
        if key not in registry:
            errors.append(f"Missing top-level key: {key}")

    entity_lookup = _entity_map(registry)

    for entity_name, entity_def in entity_lookup.items():
        has_join_key = "join_key" in entity_def
        has_join_keys = "join_keys" in entity_def

        if not (has_join_key or has_join_keys):
            errors.append(f"Entity '{entity_name}' must define 'join_key' or 'join_keys'.")

        if has_join_key and has_join_keys:
            errors.append(f"Entity '{entity_name}' must define only one of 'join_key' or 'join_keys'.")

    feature_sets = registry.get("feature_sets", [])
    for fs in feature_sets:
        fs_name = fs.get("name", "<unknown>")
        entity_name = fs.get("entity")
        if entity_name not in entity_lookup:
            errors.append(f"Feature set '{fs_name}' references unknown entity '{entity_name}'.")

        for key in ["version", "storage_path", "features"]:
            if key not in fs:
                errors.append(f"Feature set '{fs_name}' missing required key '{key}'.")

        if "features" in fs and not fs["features"]:
            errors.append(f"Feature set '{fs_name}' must include at least one feature.")

        for feature in fs.get("features", []):
            for feature_key in ["name", "dtype", "source_column", "transformation"]:
                if feature_key not in feature:
                    errors.append(
                        f"Feature set '{fs_name}' has feature missing '{feature_key}'."
                    )

    retrieval_rules = registry.get("governance", {}).get("retrieval_rules", {})
    for mode in ["training", "inference"]:
        if mode not in retrieval_rules:
            errors.append(f"Missing retrieval rules for mode '{mode}'.")
            continue

        include_sets = retrieval_rules[mode].get("include_feature_sets", [])
        known_sets = {fs.get("name") for fs in feature_sets}
        for fs_name in include_sets:
            if fs_name not in known_sets:
                errors.append(
                    f"Retrieval mode '{mode}' includes unknown feature set '{fs_name}'."
                )

    return errors


def get_feature_sets(
    registry: Dict[str, Any],
    mode: str = "training",
    version: Optional[str] = None
) -> List[Dict[str, Any]]:
    retrieval_rules = registry["governance"]["retrieval_rules"]
    allowed_sets = set(retrieval_rules[mode]["include_feature_sets"])

    if version is None:
        version = registry.get("registry_version")

    selected = []
    for fs in registry["feature_sets"]:
        if fs["name"] in allowed_sets and fs["version"] == version:
            selected.append(fs)

    return selected


def list_feature_names(
    registry: Dict[str, Any],
    mode: str = "training",
    version: Optional[str] = None
) -> Dict[str, List[str]]:
    feature_sets = get_feature_sets(registry, mode=mode, version=version)
    return {
        fs["name"]: [feature["name"] for feature in fs["features"]]
        for fs in feature_sets
    }


def get_join_keys_for_entity(
    registry: Dict[str, Any],
    entity_name: str
) -> List[str]:
    entity_lookup = _entity_map(registry)
    entity_def = entity_lookup[entity_name]

    if "join_key" in entity_def:
        return [entity_def["join_key"]]

    return entity_def["join_keys"]


def resolve_storage_path(
    feature_set: Dict[str, Any],
    registry_path: Union[str, Path]
) -> Path:
    registry_path = Path(registry_path).resolve()
    registry_dir = registry_path.parent
    storage_path = Path(feature_set["storage_path"])

    if storage_path.is_absolute():
        return storage_path

    project_root_candidate = registry_dir.parents[1] if len(registry_dir.parents) >= 2 else registry_dir
    return (project_root_candidate / storage_path).resolve()


def load_feature_data(
    feature_set: Dict[str, Any],
    registry_path: Union[str, Path]
) -> pd.DataFrame:
    storage_path = resolve_storage_path(feature_set, registry_path)

    if not storage_path.exists():
        raise FileNotFoundError(
            f"Feature data file not found for feature set '{feature_set['name']}': {storage_path}"
        )

    suffix = storage_path.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(storage_path)
    if suffix == ".csv":
        return pd.read_csv(storage_path)

    raise ValueError(
        f"Unsupported storage format '{suffix}' for feature set '{feature_set['name']}'. "
        "Use .parquet or .csv."
    )


def get_required_columns(
    registry: Dict[str, Any],
    feature_set: Dict[str, Any]
) -> List[str]:
    join_keys = get_join_keys_for_entity(registry, feature_set["entity"])
    feature_names = [f["name"] for f in feature_set["features"]]
    return join_keys + feature_names


def validate_feature_data_columns(
    registry: Dict[str, Any],
    feature_set: Dict[str, Any],
    df: pd.DataFrame
) -> None:
    required_cols = get_required_columns(registry, feature_set)
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise FeatureRegistryError(
            f"Feature data for '{feature_set['name']}' is missing required columns: {missing_cols}"
        )


def retrieve_feature_set(
    registry: Dict[str, Any],
    feature_set: Dict[str, Any],
    entity_df: pd.DataFrame,
    registry_path: Union[str, Path]
) -> pd.DataFrame:
    join_keys = get_join_keys_for_entity(registry, feature_set["entity"])
    feature_df = load_feature_data(feature_set, registry_path)
    validate_feature_data_columns(registry, feature_set, feature_df)

    for key in join_keys:
        if key not in entity_df.columns:
            raise FeatureRegistryError(
                f"Entity input is missing join key '{key}' required for feature set '{feature_set['name']}'."
            )

    selected_cols = get_required_columns(registry, feature_set)
    feature_df = feature_df[selected_cols].drop_duplicates(subset=join_keys)

    merged = entity_df.merge(feature_df, on=join_keys, how="left")
    return merged


def retrieve_features(
    registry: Dict[str, Any],
    entity_df: pd.DataFrame,
    mode: str = "training",
    version: Optional[str] = None,
    registry_path: Union[str, Path] = DEFAULT_REGISTRY_PATH
) -> pd.DataFrame:
    feature_sets = get_feature_sets(registry, mode=mode, version=version)
    result = entity_df.copy()

    for fs in feature_sets:
        result = retrieve_feature_set(registry, fs, result, registry_path)

    return result


def get_training_dataset(
    registry: Dict[str, Any],
    entity_df: pd.DataFrame,
    label_column: Optional[str] = None,
    version: Optional[str] = None,
    registry_path: Union[str, Path] = DEFAULT_REGISTRY_PATH
) -> pd.DataFrame:
    training_df = retrieve_features(
        registry=registry,
        entity_df=entity_df,
        mode="training",
        version=version,
        registry_path=registry_path,
    )

    if label_column and label_column not in training_df.columns:
        raise FeatureRegistryError(
            f"Label column '{label_column}' not found after training retrieval."
        )

    return training_df


def get_inference_dataset(
    registry: Dict[str, Any],
    entity_df: pd.DataFrame,
    version: Optional[str] = None,
    registry_path: Union[str, Path] = DEFAULT_REGISTRY_PATH
) -> pd.DataFrame:
    return retrieve_features(
        registry=registry,
        entity_df=entity_df,
        mode="inference",
        version=version,
        registry_path=registry_path,
    )


def describe_registry(
    registry: Dict[str, Any],
    mode: str = "training",
    version: Optional[str] = None
) -> pd.DataFrame:
    rows = []
    for fs in get_feature_sets(registry, mode=mode, version=version):
        join_keys = get_join_keys_for_entity(registry, fs["entity"])
        for feature in fs["features"]:
            rows.append({
                "feature_set": fs["name"],
                "entity": fs["entity"],
                "join_keys": ",".join(join_keys),
                "feature_name": feature["name"],
                "dtype": feature["dtype"],
                "source_column": feature["source_column"],
                "transformation": feature["transformation"],
                "storage_path": fs["storage_path"],
                "version": fs["version"],
            })
    return pd.DataFrame(rows)


def demo(
    registry_path: Union[str, Path] = DEFAULT_REGISTRY_PATH,
    version: Optional[str] = None
) -> None:
    registry = load_feature_registry(registry_path)

    errors = validate_registry(registry)
    if errors:
        raise FeatureRegistryError("Registry validation failed:\n- " + "\n- ".join(errors))

    if version is None:
        version = registry.get("registry_version", "v1")

    print(f"Loaded registry: {registry['registry_name']} ({version})")
    print("\nTraining features:")
    print(list_feature_names(registry, mode="training", version=version))

    print("\nInference features:")
    print(list_feature_names(registry, mode="inference", version=version))

    print("\nFeature catalog:")
    print(describe_registry(registry, mode="training", version=version).head(20).to_string(index=False))

    training_entities = pd.DataFrame({
        "user_id": [101, 102, 103],
        "item_id": [5001, 5002, 5003],
    })

    inference_entities = pd.DataFrame({
        "user_id": [101, 102],
        "item_id": [5001, 5002],
    })

    print("\nTraining entity input:")
    print(training_entities.to_string(index=False))

    try:
        training_dataset = get_training_dataset(
            registry=registry,
            entity_df=training_entities,
            version=version,
            registry_path=registry_path,
        )
        print("\nRetrieved training dataset:")
        print(training_dataset.head().to_string(index=False))
    except Exception as e:
        print(f"\nTraining retrieval demo could not complete: {e}")

    print("\nInference entity input:")
    print(inference_entities.to_string(index=False))

    try:
        inference_dataset = get_inference_dataset(
            registry=registry,
            entity_df=inference_entities,
            version=version,
            registry_path=registry_path,
        )
        print("\nRetrieved inference dataset:")
        print(inference_dataset.head().to_string(index=False))
    except Exception as e:
        print(f"\nInference retrieval demo could not complete: {e}")


if __name__ == "__main__":
    print(Path.cwd())
    demo()


C:\Users\barath\recomart-pipeline


FileNotFoundError: Registry file not found: C:\Users\feature_store\registry\feature_registry.yaml